# MiniMind Studio — Verificação automática de LoRA (Colab)
Use **Ambiente de execução → Executar tudo** (de preferência com GPU T4). O notebook:
1. valida os 4 presets LoRA do app (rank, alpha, módulos-alvo) contra o `peft`;
2. treina de verdade o preset **MiniMind** (modelo pequeno ~135M como substituto) por poucos passos;
3. confere que a loss cai, o adaptador é minúsculo e o checkpoint salva/recarrega;
4. imprime um relatório **PASS/FAIL** final.

In [ ]:
!pip -q install -U transformers peft accelerate

In [ ]:
import torch, json, os, time
from dataclasses import dataclass
# Presets espelhados de LoraViewModel.kt
PRESETS = {
 "MiniMind-LM (100M)":   dict(rank=8,  alpha=16, lr=2e-4,   epochs=3, batch=4, modules=["q_proj","v_proj"]),
 "Gemma-2-2B (2.6B)":    dict(rank=16, alpha=32, lr=1e-4,   epochs=4, batch=2, modules=["q_proj","v_proj","o_proj"]),
 "Phi-3-Mini (3.8B)":    dict(rank=16, alpha=32, lr=1.5e-4, epochs=3, batch=2, modules=["q_proj","v_proj"]),
 "Llama-3-8B (8.0B)":    dict(rank=32, alpha=64, lr=5e-5,   epochs=5, batch=1, modules=["q_proj","k_proj","v_proj","o_proj"]),
}
RESULTS = []
def check(name, ok, detail=""):
    RESULTS.append((name, bool(ok), detail)); print(("PASS" if ok else "FAIL"), "-", name, detail)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from peft import LoraConfig
for n,p in PRESETS.items():
    cfg = LoraConfig(r=p["rank"], lora_alpha=p["alpha"], target_modules=p["modules"], lora_dropout=0.05, task_type="CAUSAL_LM")
    ARCH_MODULES = {"q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"}  # nomes padrão Llama/Gemma/Phi
    check(f"preset válido: {n}", cfg.r>0 and cfg.lora_alpha>=cfg.r and set(cfg.target_modules)<=ARCH_MODULES, f"r={cfg.r} alpha={cfg.lora_alpha} módulos={sorted(cfg.target_modules)}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import get_peft_model
BASE = "HuggingFaceTB/SmolLM2-135M"  # substituto do MiniMind-LM (100M)
p = PRESETS["MiniMind-LM (100M)"]
tok = AutoTokenizer.from_pretrained(BASE); tok.pad_token = tok.pad_token or tok.eos_token
dev = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32).to(dev)
model = get_peft_model(model, LoraConfig(r=p["rank"], lora_alpha=p["alpha"], target_modules=p["modules"], lora_dropout=0.05, task_type="CAUSAL_LM"))
tr = sum(x.numel() for x in model.parameters() if x.requires_grad); tot = sum(x.numel() for x in model.parameters())
check("apenas LoRA é treinável (<5% dos parâmetros)", tr/tot < 0.05, f"{tr:,}/{tot:,} = {100*tr/tot:.2f}%")

In [ ]:
texts = ["Vibração acima de 8 mm/s na turbina 4 indica risco de falha; acione manutenção preventiva."]*16
enc = tok(texts, return_tensors="pt", padding=True).to(dev)
opt = torch.optim.AdamW([x for x in model.parameters() if x.requires_grad], lr=p["lr"])
model.train(); losses=[]
for step in range(p["epochs"]*10):
    i = (step*p["batch"]) % 16
    b = {k:v[i:i+p["batch"]] for k,v in enc.items()}
    lab = b["input_ids"].masked_fill(b["attention_mask"]==0, -100); out = model(**b, labels=lab); out.loss.backward(); opt.step(); opt.zero_grad(); losses.append(out.loss.item())
print([round(l,3) for l in losses[::5]])
check("loss diminui durante o treino", sum(losses[-5:])/5 < sum(losses[:5])/5, f"{losses[0]:.3f} -> {losses[-1]:.3f}")

In [ ]:
model.save_pretrained("adapter_minimind")
size = sum(os.path.getsize(os.path.join("adapter_minimind",f)) for f in os.listdir("adapter_minimind"))/1e6
check("adaptador salvo e pequeno (<20 MB)", size<20, f"{size:.2f} MB")
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32).to(dev)
reloaded = PeftModel.from_pretrained(base, "adapter_minimind").eval()
model.eval(); probe = tok("Vibração crítica detectada:", return_tensors="pt").to(dev)
with torch.no_grad(): diff = (model(**probe).logits - reloaded(**probe).logits).abs().max().item()
check("adaptador recarregado é idêntico ao treinado (logits)", diff < 1e-4, f"max|diff|={diff:.2e}")
gen = reloaded.generate(**probe, max_new_tokens=20)
check("adaptador gera tokens novos", gen.shape[1] > probe["input_ids"].shape[1], tok.decode(gen[0], skip_special_tokens=True)[:80])

In [ ]:
print("\n=== RELATÓRIO FINAL ===")
for n,ok,d in RESULTS: print("[%s] %s %s" % ("PASS" if ok else "FAIL", n, d))
failed=[n for n,ok,_ in RESULTS if not ok]
print("\nRESULTADO:", "TUDO OK ✅" if not failed else f"{len(failed)} falha(s) ❌")
assert not failed, failed